# The same week, from a file

`data/week.csv` stores the week. The first column is `name`. `read_csv` brings that column in as data, with a default index of 0, 1, 2, 3. The five day columns still sum to 89.

`index_col="name"` makes the reader the index, which is what `loc["Ada"]` expects. After that, the loaded table equals the table built in memory.

`data/prices.csv`, `data/purchases.csv`, and `data/roster.csv` are the bill. Read together, the inner-join spend is again 38, and Edsger is the roster name absent from the purchases.


In [1]:
import pandas as pd
# Locate the desk files beside this notebook, or under the course root.
from pathlib import Path

def data_path(name):
    here = Path.cwd()
    for folder in [here, *here.parents]:
        nested = folder / "04-Pandas" / "data" / name
        direct = folder / "data" / name
        if nested.exists():
            return nested
        if direct.exists() and (folder / "data" / "week.csv").exists():
            return direct
    raise FileNotFoundError(name)
# Read the week two ways, then the bill from the other three files.
plain = pd.read_csv(data_path("week.csv"))
indexed = pd.read_csv(data_path("week.csv"), index_col="name")
built = pd.DataFrame(
    {
        "Mon": [4, 6, 3, 5],
        "Tue": [5, 6, 0, 7],
        "Wed": [3, 6, 8, 2],
        "Thu": [6, 6, 1, 4],
        "Fri": [2, 6, 4, 5],
    },
    index=["Ada", "Grace", "Alan", "Linus"],
)
print(list(plain.columns), int(indexed.sum().sum()), indexed.equals(built))
assert list(plain.columns) == ["name", "Mon", "Tue", "Wed", "Thu", "Fri"]
assert int(plain[["Mon", "Tue", "Wed", "Thu", "Fri"]].sum().sum()) == 89
assert indexed.equals(built)
assert int(indexed.loc["Alan", "Tue"]) == 0
prices = pd.read_csv(data_path("prices.csv"))
purchases = pd.read_csv(data_path("purchases.csv"))
roster = pd.read_csv(data_path("roster.csv"))
bill = purchases.merge(prices, on="item", how="inner")
spend = int((bill["qty"] * bill["price"]).sum())
missing = sorted(set(roster["name"]) - set(purchases["name"]))
print(spend, missing)
assert spend == 38
assert missing == ["Edsger"]
assert int(prices.loc[prices["item"] == "grid", "price"].iloc[0]) == 4


['name', 'Mon', 'Tue', 'Wed', 'Thu', 'Fri'] 89 True
38 ['Edsger']


## Pitfall

Summing a frame that still has the `name` column as text would fail, or would ignore that column, depending on the call. The page total is the sum of the five day columns. Alan's Tuesday is the integer 0 in the file. A blank cell would have been an empty field, and this file has no empty field.

## Summary

`week.csv` has a name column and five day columns, totaling 89. With `index_col="name"` it matches the table built in memory. The purchase files bill 38, and Edsger is the roster name with no purchase row.
